In [1]:
import numpy as np
from scipy.integrate import solve_ivp as sp_solve_ivp
from scipy.integrate import odeint
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from typing import List
device = 'cpu'
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'serif'

from ftnode.utils import set_global_seed
from ftnode.node import (
    FTNODE, FeluSigmoidMLP, GeluSigmoidMLP,)

import torchode

seed = 1234
set_global_seed(seed)

random_state = 1234

[Seed] Deterministic mode enabled (may reduce speed).


## Generate Data

In [2]:
## Define system 
gamma = 50
def sigmoid(x,gamma=gamma):
    return 1 / (1+np.exp(-gamma*x))

eps = 0.02
q1, q2 = (0.08, 0.04)
b1 = 1-eps
b2 = 1-eps

def c1_in(x):
    return q1*(1-sigmoid(x-b1))

def c2_in(y):
    return q1*(1-sigmoid(y-b2))

def c1_out(y):
    return q2*(1-sigmoid(y-b2))

def c2_out(y):
    return q2

def two_tank_system(t,x,u):
    x1, x2 = x
    p, v = u
    x1= np.maximum(x1,0)
    x2 = np.maximum(x2,0)
    dx1dt = c1_in(x1)*(1-v)*p-c1_out(x2)*np.sqrt(x1)
    dx2dt = c2_in(x2)*v*p +c1_out(x2)*np.sqrt(x1)-q2*np.sqrt(x2)
    return np.hstack([dx1dt,dx2dt])


p_vals = np.linspace(0,1,101)
v_vals = np.linspace(0,1,101)

In [3]:
t_max = 200
n_colloc = 501

x0s = np.linspace(0,1,21)

In [4]:
p_train = p_vals[10:-10:10]
v_train = v_vals[10:-10:10]

In [5]:
Xs = [] 
Us = []
t = np.linspace(0,t_max, n_colloc)
for pi in tqdm(p_train):
    for vi in v_train:
        
        for x0 in zip(x0s, x0s):
            u = np.array([pi,vi])

            sol = sp_solve_ivp(
                two_tank_system,
                t_span = [0,t_max],
                y0 = np.array(x0),
                t_eval= np.linspace(0,t_max, n_colloc),
                args =(u,)
            )

            Xs.append(sol.y.T)
            Us.append((pi,vi))

Us = np.array(Us)
Xs = np.array(Xs)

  0%|          | 0/9 [00:00<?, ?it/s]

In [6]:
dXs = np.zeros_like(Xs)
T = t[np.newaxis,:,np.newaxis]
X_diff = Xs[:,2:,:] - Xs[:,:-2,:]
T_diff = T[:,2:,:] - T[:,:-2,:]

dXs[:,1:-1,:] = X_diff/T_diff
dXs[:,0,:] = (Xs[:,1,:] - Xs[:,0,:]) / (T[:,1,:] - T[:,0,:])
dXs[:,-1,:] = (Xs[:,-1,:] - Xs[:,-2,:]) / (T[:,-1,:] - T[:,-2,:])

In [7]:
dX_tensor = [
    torch.tensor(dxi,dtype=torch.float32,device=device) for dxi in dXs
]
X_tensor = [
    torch.tensor(xi,dtype=torch.float32,device=device) for xi in Xs
]
U_tensor = [
    torch.tensor(ui,dtype=torch.float32,device=device) for ui in Us
]

T_tensor = [
    torch.tensor(t,dtype=torch.float32, device=device) for _ in range(len(Xs))
]

In [8]:
class GradDataset(torch.utils.data.Dataset):
    def __init__(self, dX: List, X: List, T: List, U: List):
        self.dX = dX
        self.X = X
        self.T = T
        self.U = U

    def __len__(self):
        return len(self.dX)

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError(
                f"Index {idx} is out of bounds of dataset size: {len(self)}."
            )

        dXi = self.dX[idx]
        Xi = self.X[idx]
        ti = self.T[idx]
        ui = self.U[idx]

        return dXi, Xi, ti, ui

dataset = GradDataset(dX = dX_tensor,X = X_tensor, T = T_tensor, U = U_tensor)
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=50, shuffle=True)

## Run $k$-Folds Cross-validation

In [9]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, SubsetRandomSampler
import time
import copy

In [10]:
avg_best_val_losses = []
avg_best_train_losses = []

In [11]:
# --- Configuration ---
k_folds = 5
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)

val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,10,10,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,10,10,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)


--- FOLD 1/5 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.4743e-02, Val Loss = 1.5565e-03, Time = 3.7932e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 7.0982e-04, Val Loss = 3.4782e-04, Time = 2.1290e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 2.2798e-04, Val Loss = 1.5105e-04, Time = 2.1678e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 1.1139e-04, Val Loss = 8.1230e-05, Time = 2.0680e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 6.8424e-05, Val Loss = 5.4657e-05, Time = 2.0954e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.8600e-05, Val Loss = 4.1132e-05, Time = 2.1074e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 8.4784e-06, Val Loss = 8.1182e-06, Time = 2.1426e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.5667e-06, Val Loss = 3.4539e-06, Time = 2.4974e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 3.0736e-06, Val Loss = 3.0432e-06, Time = 2.1221e-01, lr = 5.0000e-03
Epoch 400: Train Loss = 2.9501e-06, Val Loss = 2.8911e-06, Time = 2.1047e-01, lr = 1.5625e-04
Epoch 500: Train Loss = 3.0122e-06, Val Loss = 2.9034e-06, Time = 2.2015

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.6479e-02, Val Loss = 2.0218e-03, Time = 3.1002e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 6.5952e-04, Val Loss = 2.3796e-04, Time = 2.2222e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.7544e-04, Val Loss = 1.1587e-04, Time = 2.3395e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 1.0224e-04, Val Loss = 8.0483e-05, Time = 2.4705e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.8836e-05, Val Loss = 6.5524e-05, Time = 2.2390e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 6.5493e-05, Val Loss = 5.6800e-05, Time = 2.1597e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 7.4581e-06, Val Loss = 7.5425e-06, Time = 2.1400e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.6215e-06, Val Loss = 3.8367e-06, Time = 2.2514e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 3.0011e-06, Val Loss = 3.1836e-06, Time = 2.2102e-01, lr = 1.0000e-02
Epoch 400: Train Loss = 2.8336e-06, Val Loss = 3.0257e-06, Time = 2.1381e-01, lr = 1.2500e-03
Epoch 500: Train Loss = 2.1417e-06, Val Loss = 2.1848e-06, Time = 2.1244

...

Epoch 0: Train Loss = 2.1233e-02, Val Loss = 3.0220e-03, Time = 2.1446e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 9.1091e-04, Val Loss = 2.0116e-04, Time = 2.0722e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.5066e-04, Val Loss = 1.1435e-04, Time = 2.0816e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 9.4375e-05, Val Loss = 8.5424e-05, Time = 2.1239e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.2433e-05, Val Loss = 6.9269e-05, Time = 2.1792e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 5.9719e-05, Val Loss = 5.7065e-05, Time = 2.0863e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 8.1479e-06, Val Loss = 8.0288e-06, Time = 2.1014e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 4.0141e-06, Val Loss = 4.2754e-06, Time = 2.0504e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 3.2130e-06, Val Loss = 3.1912e-06, Time = 2.0985e-01, lr = 1.0000e-02
Epoch 400: Train Loss = 1.3682e-06, Val Loss = 1.5474e-06, Time = 2.0496e-01, lr = 1.0000e-02
Epoch 500: Train Loss = 1.0354e-06, Val Loss = 1.0420e-06, Time = 2.0405

Fold 4:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.4486e-02, Val Loss = 4.0222e-03, Time = 2.2133e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.5188e-03, Val Loss = 7.4811e-04, Time = 2.0304e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 4.5875e-04, Val Loss = 2.9306e-04, Time = 2.1294e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 2.1500e-04, Val Loss = 1.5633e-04, Time = 2.0942e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 1.2425e-04, Val Loss = 1.0119e-04, Time = 2.1098e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 8.9134e-05, Val Loss = 7.6289e-05, Time = 2.0652e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 1.0749e-05, Val Loss = 1.0072e-05, Time = 2.0823e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.8735e-06, Val Loss = 3.8237e-06, Time = 2.2205e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 3.3517e-06, Val Loss = 3.3291e-06, Time = 2.1058e-01, lr = 5.0000e-03
Epoch 400: Train Loss = 3.1589e-06, Val Loss = 2.9985e-06, Time = 2.3015e-01, lr = 2.5000e-03
Epoch 500: Train Loss = 2.9273e-06, Val Loss = 2.8456e-06, Time = 2.1056

Fold 5:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.7968e-02, Val Loss = 1.7444e-03, Time = 2.1674e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 7.9956e-04, Val Loss = 3.2960e-04, Time = 2.1181e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 2.3384e-04, Val Loss = 1.7633e-04, Time = 2.0867e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 1.3221e-04, Val Loss = 1.1211e-04, Time = 2.0944e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 9.5948e-05, Val Loss = 8.7254e-05, Time = 2.0877e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 7.7010e-05, Val Loss = 7.3781e-05, Time = 2.0127e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 9.2166e-06, Val Loss = 9.2957e-06, Time = 2.0817e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 4.5818e-06, Val Loss = 4.4931e-06, Time = 2.0474e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 3.1442e-06, Val Loss = 3.0316e-06, Time = 2.1549e-01, lr = 1.0000e-02
Epoch 400: Train Loss = 2.9182e-06, Val Loss = 2.8397e-06, Time = 2.0875e-01, lr = 2.5000e-03
Epoch 500: Train Loss = 2.8419e-06, Val Loss = 2.8074e-06, Time = 2.0627

In [12]:
# --- Configuration ---
k_folds = 5
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,10,10,10,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,10,10,10,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)


--- FOLD 1/5 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.8552e-02, Val Loss = 1.1419e-03, Time = 4.2339e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 5.8042e-04, Val Loss = 3.3557e-04, Time = 2.8827e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 2.5539e-04, Val Loss = 1.9834e-04, Time = 2.7806e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 1.4226e-04, Val Loss = 1.0133e-04, Time = 2.7615e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 8.5884e-05, Val Loss = 6.9956e-05, Time = 2.6909e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 6.4285e-05, Val Loss = 5.4858e-05, Time = 2.8562e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.0342e-06, Val Loss = 3.7494e-06, Time = 2.7651e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.3669e-06, Val Loss = 3.1243e-06, Time = 4.2006e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.6715e-06, Val Loss = 2.6750e-06, Time = 3.4625e-01, lr = 2.5000e-03
Epoch 400: Train Loss = 2.2624e-06, Val Loss = 2.2555e-06, Time = 4.2865e-01, lr = 2.5000e-03
Epoch 500: Train Loss = 1.1005e-06, Val Loss = 1.0637e-06, Time = 3.4780

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.3409e-02, Val Loss = 2.5183e-03, Time = 3.5164e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 9.3051e-04, Val Loss = 4.6727e-04, Time = 3.4934e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 3.1466e-04, Val Loss = 2.2178e-04, Time = 3.4351e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 1.7238e-04, Val Loss = 1.2444e-04, Time = 3.3952e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 1.0804e-04, Val Loss = 8.5419e-05, Time = 3.4644e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 8.0866e-05, Val Loss = 6.5048e-05, Time = 3.5049e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 5.4140e-06, Val Loss = 5.4430e-06, Time = 3.4082e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.0424e-06, Val Loss = 3.4657e-06, Time = 3.3795e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.7898e-06, Val Loss = 2.9149e-06, Time = 3.4563e-01, lr = 1.2500e-03
Epoch 400: Train Loss = 2.6921e-06, Val Loss = 2.8793e-06, Time = 3.6050e-01, lr = 3.9063e-05
Epoch 500: Train Loss = 2.6784e-06, Val Loss = 2.9103e-06, Time = 3.3882

...

Epoch 0: Train Loss = 1.8544e-02, Val Loss = 1.0916e-03, Time = 3.4516e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 4.3549e-04, Val Loss = 1.8742e-04, Time = 4.7507e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.3039e-04, Val Loss = 8.6067e-05, Time = 3.9499e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 7.1235e-05, Val Loss = 6.7276e-05, Time = 3.6291e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.6244e-05, Val Loss = 5.4218e-05, Time = 3.5778e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.4959e-05, Val Loss = 4.4583e-05, Time = 3.4789e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.6861e-06, Val Loss = 3.6883e-06, Time = 3.5505e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.4383e-06, Val Loss = 2.8946e-06, Time = 3.6106e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.3060e-06, Val Loss = 2.3471e-06, Time = 3.5125e-01, lr = 1.2500e-03
Epoch 400: Train Loss = 2.2355e-06, Val Loss = 2.3052e-06, Time = 3.5614e-01, lr = 1.9531e-05
Epoch 500: Train Loss = 2.3022e-06, Val Loss = 2.3023e-06, Time = 3.4122

Fold 4:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.4921e-02, Val Loss = 1.6028e-03, Time = 3.6217e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 5.8415e-04, Val Loss = 2.2178e-04, Time = 3.5782e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.6897e-04, Val Loss = 1.1303e-04, Time = 3.5659e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 9.5045e-05, Val Loss = 7.5543e-05, Time = 3.5147e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.1667e-05, Val Loss = 6.1755e-05, Time = 3.4234e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 6.0076e-05, Val Loss = 5.5068e-05, Time = 3.5036e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.8302e-06, Val Loss = 4.8181e-06, Time = 3.6002e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.2697e-06, Val Loss = 3.2005e-06, Time = 3.5416e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.6184e-06, Val Loss = 2.5908e-06, Time = 3.5371e-01, lr = 5.0000e-03
Epoch 400: Train Loss = 2.3082e-06, Val Loss = 2.1765e-06, Time = 3.6440e-01, lr = 5.0000e-03
Epoch 500: Train Loss = 1.1740e-06, Val Loss = 1.1916e-06, Time = 3.8622

Fold 5:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 3.1739e-02, Val Loss = 1.5556e-02, Time = 3.6645e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 6.5670e-03, Val Loss = 1.9313e-03, Time = 3.6064e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 8.6509e-04, Val Loss = 4.9037e-04, Time = 3.4475e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 4.0048e-04, Val Loss = 3.5713e-04, Time = 4.3489e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 2.8723e-04, Val Loss = 2.3138e-04, Time = 3.5527e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 1.5829e-04, Val Loss = 1.0671e-04, Time = 3.3640e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.9245e-06, Val Loss = 3.9358e-06, Time = 3.4485e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.2890e-06, Val Loss = 2.4422e-06, Time = 3.4453e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 1.5760e-06, Val Loss = 1.7602e-06, Time = 3.7980e-01, lr = 5.0000e-03
Epoch 400: Train Loss = 1.0762e-06, Val Loss = 1.1079e-06, Time = 3.4223e-01, lr = 1.2500e-03
Epoch 500: Train Loss = 9.9934e-07, Val Loss = 1.0057e-06, Time = 3.4502

In [13]:
# --- Configuration ---
k_folds = 5
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []


def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,20,20,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,20,20,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)


--- FOLD 1/5 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.3502e-02, Val Loss = 5.5496e-04, Time = 4.6253e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 2.9076e-04, Val Loss = 1.7641e-04, Time = 4.5523e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.2900e-04, Val Loss = 9.9280e-05, Time = 4.3313e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 8.3603e-05, Val Loss = 6.8302e-05, Time = 4.5331e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 6.1838e-05, Val Loss = 5.4042e-05, Time = 4.4508e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 5.1239e-05, Val Loss = 4.6229e-05, Time = 4.3540e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 6.4844e-06, Val Loss = 6.3021e-06, Time = 4.4409e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.1845e-06, Val Loss = 3.1351e-06, Time = 4.6185e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.9955e-06, Val Loss = 2.9006e-06, Time = 4.3625e-01, lr = 6.2500e-04
Epoch 400: Train Loss = 2.9118e-06, Val Loss = 2.8893e-06, Time = 4.3131e-01, lr = 4.8828e-06
Epoch 500: Train Loss = 2.9281e-06, Val Loss = 2.8792e-06, Time = 4.6758

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 7.8113e-03, Val Loss = 4.6348e-04, Time = 4.4184e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 2.4345e-04, Val Loss = 1.2229e-04, Time = 4.3271e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 8.6164e-05, Val Loss = 5.6123e-05, Time = 4.5316e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 4.7237e-05, Val Loss = 3.6327e-05, Time = 4.4908e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 3.2010e-05, Val Loss = 2.3527e-05, Time = 4.7416e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 2.0977e-05, Val Loss = 1.7092e-05, Time = 4.4550e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.2815e-06, Val Loss = 4.3608e-06, Time = 4.2620e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.9628e-06, Val Loss = 3.1712e-06, Time = 4.4337e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.7358e-06, Val Loss = 2.8050e-06, Time = 4.6308e-01, lr = 5.0000e-03
Epoch 400: Train Loss = 2.2612e-06, Val Loss = 2.4515e-06, Time = 5.0748e-01, lr = 6.2500e-04
Epoch 500: Train Loss = 2.2437e-06, Val Loss = 2.4113e-06, Time = 4.5159

...

Epoch 0: Train Loss = 9.4707e-03, Val Loss = 5.8621e-04, Time = 4.5139e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 2.9792e-04, Val Loss = 1.5843e-04, Time = 4.5232e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.1853e-04, Val Loss = 9.2565e-05, Time = 4.3894e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 7.8476e-05, Val Loss = 7.5904e-05, Time = 4.4019e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 6.7419e-05, Val Loss = 6.7291e-05, Time = 4.4540e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 5.8612e-05, Val Loss = 5.8815e-05, Time = 4.4585e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.6383e-06, Val Loss = 4.6322e-06, Time = 5.0542e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.0702e-06, Val Loss = 3.0831e-06, Time = 4.3774e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.8347e-06, Val Loss = 2.9523e-06, Time = 4.5239e-01, lr = 2.5000e-03
Epoch 400: Train Loss = 2.7310e-06, Val Loss = 2.7962e-06, Time = 4.3274e-01, lr = 3.1250e-04
Epoch 500: Train Loss = 2.7202e-06, Val Loss = 2.7741e-06, Time = 4.4052

Fold 4:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.0874e-02, Val Loss = 8.3467e-04, Time = 4.5599e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 3.4847e-04, Val Loss = 1.4557e-04, Time = 4.3934e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.0802e-04, Val Loss = 7.0423e-05, Time = 4.3586e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.1332e-05, Val Loss = 5.1288e-05, Time = 4.3520e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 4.9100e-05, Val Loss = 4.3196e-05, Time = 4.4000e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.1128e-05, Val Loss = 3.6409e-05, Time = 4.4905e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 5.0363e-06, Val Loss = 4.9365e-06, Time = 4.4628e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.2732e-06, Val Loss = 3.0444e-06, Time = 4.2813e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.6676e-06, Val Loss = 2.5213e-06, Time = 4.5350e-01, lr = 1.0000e-02
Epoch 400: Train Loss = 2.3130e-06, Val Loss = 2.2568e-06, Time = 4.5120e-01, lr = 5.0000e-03
Epoch 500: Train Loss = 6.6413e-07, Val Loss = 7.5110e-07, Time = 4.5784

Fold 5:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.2134e-02, Val Loss = 3.7231e-04, Time = 4.9041e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.6483e-04, Val Loss = 1.2534e-04, Time = 5.0378e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 8.1189e-05, Val Loss = 7.2358e-05, Time = 4.5795e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.4389e-05, Val Loss = 6.2274e-05, Time = 4.5958e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.5277e-05, Val Loss = 5.3818e-05, Time = 4.4572e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.9459e-05, Val Loss = 4.6846e-05, Time = 4.7334e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 7.3139e-06, Val Loss = 7.8904e-06, Time = 4.7160e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.1616e-06, Val Loss = 3.0918e-06, Time = 4.4895e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.7283e-06, Val Loss = 2.7066e-06, Time = 4.4170e-01, lr = 5.0000e-03
Epoch 400: Train Loss = 1.2919e-06, Val Loss = 1.2302e-06, Time = 4.3690e-01, lr = 2.5000e-03
Epoch 500: Train Loss = 8.0592e-07, Val Loss = 8.1928e-07, Time = 4.3920

In [14]:
# --- Configuration ---
k_folds = 5
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,50,50,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,50,50,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)


--- FOLD 1/5 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 6.0730e-03, Val Loss = 2.4122e-04, Time = 9.1800e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.4321e-04, Val Loss = 8.0737e-05, Time = 7.8682e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 6.1923e-05, Val Loss = 4.3884e-05, Time = 7.6334e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 3.7657e-05, Val Loss = 2.8967e-05, Time = 8.2763e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 2.3073e-05, Val Loss = 1.9114e-05, Time = 7.9438e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 1.6632e-05, Val Loss = 1.5318e-05, Time = 7.4379e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.5167e-06, Val Loss = 3.4831e-06, Time = 7.6493e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.8825e-06, Val Loss = 2.8352e-06, Time = 7.9122e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 2.7055e-06, Val Loss = 2.6011e-06, Time = 7.5779e-01, lr = 2.5000e-03
Epoch 400: Train Loss = 2.3589e-06, Val Loss = 2.2032e-06, Time = 7.7003e-01, lr = 2.5000e-03
Epoch 500: Train Loss = 2.0307e-06, Val Loss = 1.9242e-06, Time = 7.9266

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 5.8486e-03, Val Loss = 7.9698e-05, Time = 7.7393e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 6.0743e-05, Val Loss = 3.7843e-05, Time = 8.5583e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 3.0784e-05, Val Loss = 2.2421e-05, Time = 7.9648e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 2.0402e-05, Val Loss = 1.6951e-05, Time = 7.8094e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 1.6801e-05, Val Loss = 1.4947e-05, Time = 7.6830e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 1.5042e-05, Val Loss = 1.4070e-05, Time = 7.5557e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.2877e-06, Val Loss = 4.4170e-06, Time = 7.9881e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.8249e-06, Val Loss = 2.9723e-06, Time = 8.1733e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.1042e-06, Val Loss = 2.2393e-06, Time = 7.2350e-01, lr = 1.0000e-02
Epoch 400: Train Loss = 1.8053e-06, Val Loss = 2.1328e-06, Time = 7.2849e-01, lr = 5.0000e-03
Epoch 500: Train Loss = 1.7392e-06, Val Loss = 1.9300e-06, Time = 7.7860

...

Epoch 0: Train Loss = 6.2504e-03, Val Loss = 1.2702e-04, Time = 9.5636e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 8.0879e-05, Val Loss = 6.1459e-05, Time = 7.6824e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 4.6680e-05, Val Loss = 4.3627e-05, Time = 7.9301e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 3.5396e-05, Val Loss = 3.3674e-05, Time = 7.8843e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 2.8276e-05, Val Loss = 2.6357e-05, Time = 7.7350e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 2.2052e-05, Val Loss = 2.1011e-05, Time = 7.5560e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.7414e-06, Val Loss = 3.7052e-06, Time = 8.1327e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.9340e-06, Val Loss = 2.9316e-06, Time = 7.5578e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.7922e-06, Val Loss = 2.7899e-06, Time = 7.3359e-01, lr = 6.2500e-04
Epoch 400: Train Loss = 2.7157e-06, Val Loss = 2.7880e-06, Time = 8.4272e-01, lr = 9.7656e-06
Epoch 500: Train Loss = 2.7363e-06, Val Loss = 2.7869e-06, Time = 7.7206

Fold 4:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 6.2252e-03, Val Loss = 2.2520e-04, Time = 7.8850e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.2824e-04, Val Loss = 6.9668e-05, Time = 7.5610e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 5.8663e-05, Val Loss = 4.6802e-05, Time = 8.8582e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 4.2745e-05, Val Loss = 3.6512e-05, Time = 7.9590e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 3.2304e-05, Val Loss = 2.6293e-05, Time = 8.1976e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 2.2429e-05, Val Loss = 1.8962e-05, Time = 7.6738e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.1852e-06, Val Loss = 4.3606e-06, Time = 7.4963e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.7943e-06, Val Loss = 2.7805e-06, Time = 7.6240e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.2168e-06, Val Loss = 2.0814e-06, Time = 7.3695e-01, lr = 5.0000e-03
Epoch 400: Train Loss = 1.9479e-06, Val Loss = 1.8637e-06, Time = 7.7065e-01, lr = 2.5000e-03
Epoch 500: Train Loss = 1.9139e-06, Val Loss = 1.8130e-06, Time = 9.7241

Fold 5:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 6.6362e-03, Val Loss = 2.4170e-04, Time = 7.8087e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.3505e-04, Val Loss = 6.9517e-05, Time = 8.1588e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 6.2908e-05, Val Loss = 5.7326e-05, Time = 7.5478e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 4.8767e-05, Val Loss = 4.5674e-05, Time = 7.3737e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 3.9094e-05, Val Loss = 3.5516e-05, Time = 7.5467e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 2.9996e-05, Val Loss = 2.5327e-05, Time = 8.0185e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.5982e-06, Val Loss = 4.5205e-06, Time = 7.4866e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.1377e-06, Val Loss = 3.0306e-06, Time = 7.4326e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.7659e-06, Val Loss = 2.7423e-06, Time = 7.7194e-01, lr = 6.2500e-04
Epoch 400: Train Loss = 2.7093e-06, Val Loss = 2.7134e-06, Time = 8.1933e-01, lr = 1.9531e-05
Epoch 500: Train Loss = 2.7112e-06, Val Loss = 2.6865e-06, Time = 7.6720

In [15]:
# --- Configuration ---
k_folds = 5
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,20,20,20,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,20,20,20,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)


--- FOLD 1/5 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.1864e-02, Val Loss = 1.9812e-04, Time = 7.3229e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.3473e-04, Val Loss = 8.5847e-05, Time = 6.4612e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 6.6733e-05, Val Loss = 5.1533e-05, Time = 6.0875e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 4.9335e-05, Val Loss = 4.1311e-05, Time = 6.2382e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 3.8481e-05, Val Loss = 3.3130e-05, Time = 7.0899e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 2.9495e-05, Val Loss = 2.5964e-05, Time = 6.5892e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.7433e-06, Val Loss = 3.2793e-06, Time = 1.4832e+00, lr = 1.0000e-02
Epoch 200: Train Loss = 2.7845e-06, Val Loss = 2.6942e-06, Time = 6.3224e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.3428e-06, Val Loss = 2.2955e-06, Time = 6.0351e-01, lr = 2.5000e-03
Epoch 400: Train Loss = 1.9432e-06, Val Loss = 1.9500e-06, Time = 6.2182e-01, lr = 2.5000e-03
Epoch 500: Train Loss = 1.8688e-06, Val Loss = 1.7986e-06, Time = 6.0472

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.1255e-02, Val Loss = 2.8054e-04, Time = 6.5712e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.8268e-04, Val Loss = 8.5377e-05, Time = 6.2137e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 6.9725e-05, Val Loss = 5.4265e-05, Time = 6.1503e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 5.1374e-05, Val Loss = 3.9796e-05, Time = 6.0528e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 3.4831e-05, Val Loss = 2.5855e-05, Time = 5.9878e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 2.2854e-05, Val Loss = 1.8227e-05, Time = 6.2227e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.7493e-06, Val Loss = 3.8641e-06, Time = 6.0092e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.7493e-06, Val Loss = 2.8923e-06, Time = 7.0496e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.1865e-06, Val Loss = 2.3353e-06, Time = 6.0503e-01, lr = 1.2500e-03
Epoch 400: Train Loss = 1.8878e-06, Val Loss = 2.0599e-06, Time = 6.0605e-01, lr = 1.2500e-03
Epoch 500: Train Loss = 1.7802e-06, Val Loss = 1.9902e-06, Time = 6.1007

...

Epoch 0: Train Loss = 1.2076e-02, Val Loss = 2.3405e-04, Time = 8.3825e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.7499e-04, Val Loss = 1.3264e-04, Time = 6.6931e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.0180e-04, Val Loss = 9.3492e-05, Time = 6.4246e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 8.2194e-05, Val Loss = 8.5155e-05, Time = 6.3763e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.2747e-05, Val Loss = 7.5246e-05, Time = 6.3658e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 6.3242e-05, Val Loss = 6.6278e-05, Time = 6.1086e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.3576e-06, Val Loss = 3.4220e-06, Time = 6.1726e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.7950e-06, Val Loss = 2.8912e-06, Time = 6.1442e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 2.6136e-06, Val Loss = 2.5918e-06, Time = 6.3613e-01, lr = 6.2500e-04
Epoch 400: Train Loss = 2.5529e-06, Val Loss = 2.5592e-06, Time = 8.3871e-01, lr = 9.7656e-06
Epoch 500: Train Loss = 2.6131e-06, Val Loss = 2.5913e-06, Time = 6.3699

Fold 4:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.2858e-02, Val Loss = 8.4270e-04, Time = 6.7686e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 4.9810e-04, Val Loss = 2.3009e-04, Time = 6.5365e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.3050e-04, Val Loss = 7.1411e-05, Time = 6.5268e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.2211e-05, Val Loss = 5.1641e-05, Time = 6.6614e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 4.5034e-05, Val Loss = 3.6796e-05, Time = 6.4672e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.1808e-05, Val Loss = 2.5713e-05, Time = 6.3416e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.2820e-06, Val Loss = 3.2194e-06, Time = 6.5067e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.7549e-06, Val Loss = 2.6456e-06, Time = 7.4164e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.5587e-06, Val Loss = 2.4528e-06, Time = 5.8938e-01, lr = 1.2500e-03
Epoch 400: Train Loss = 2.3188e-06, Val Loss = 2.2074e-06, Time = 6.2286e-01, lr = 6.2500e-04
Epoch 500: Train Loss = 1.9541e-06, Val Loss = 1.8818e-06, Time = 6.4866

Fold 5:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.1000e-02, Val Loss = 9.2685e-04, Time = 6.0579e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 4.5939e-04, Val Loss = 2.3433e-04, Time = 6.0718e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.2234e-04, Val Loss = 6.5876e-05, Time = 6.0312e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 4.6820e-05, Val Loss = 3.6757e-05, Time = 5.9481e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 2.6242e-05, Val Loss = 2.0184e-05, Time = 5.9242e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 1.7361e-05, Val Loss = 1.5593e-05, Time = 6.2104e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.2533e-06, Val Loss = 3.8331e-06, Time = 6.1009e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.4046e-06, Val Loss = 2.4058e-06, Time = 6.1203e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.5218e-06, Val Loss = 2.5149e-06, Time = 5.9277e-01, lr = 6.2500e-04
Epoch 400: Train Loss = 2.5090e-06, Val Loss = 2.4628e-06, Time = 6.0757e-01, lr = 1.2207e-06
Epoch 500: Train Loss = 2.5345e-06, Val Loss = 2.4529e-06, Time = 6.1759

In [16]:
# --- Configuration ---
k_folds = 5
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,10,10,10,10,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,10,10,10,10,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)


--- FOLD 1/5 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.3152e-02, Val Loss = 1.1965e-03, Time = 5.6956e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 8.2583e-04, Val Loss = 4.6066e-04, Time = 4.3953e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 2.0127e-04, Val Loss = 8.0136e-05, Time = 4.4409e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.3039e-05, Val Loss = 4.3937e-05, Time = 4.3776e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 3.9817e-05, Val Loss = 3.4140e-05, Time = 4.4988e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.1124e-05, Val Loss = 2.7409e-05, Time = 4.4334e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.3197e-06, Val Loss = 3.5905e-06, Time = 4.4340e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.6527e-06, Val Loss = 2.6271e-06, Time = 4.3269e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.5987e-06, Val Loss = 2.5070e-06, Time = 4.2396e-01, lr = 6.2500e-04
Epoch 400: Train Loss = 2.5398e-06, Val Loss = 2.4518e-06, Time = 4.3683e-01, lr = 9.7656e-06
Epoch 500: Train Loss = 2.4965e-06, Val Loss = 2.4880e-06, Time = 4.4612

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.2329e-02, Val Loss = 1.2411e-03, Time = 4.4736e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 4.5132e-04, Val Loss = 2.4792e-04, Time = 4.3835e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.6333e-04, Val Loss = 1.0251e-04, Time = 5.5079e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 9.1757e-05, Val Loss = 7.5527e-05, Time = 4.6791e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.2371e-05, Val Loss = 6.2516e-05, Time = 5.1573e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 5.8656e-05, Val Loss = 4.9376e-05, Time = 4.2683e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.4695e-06, Val Loss = 3.6792e-06, Time = 5.3228e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.6045e-06, Val Loss = 2.7978e-06, Time = 4.8904e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.3608e-06, Val Loss = 2.5632e-06, Time = 5.0733e-01, lr = 1.2500e-03
Epoch 400: Train Loss = 2.2430e-06, Val Loss = 2.4050e-06, Time = 4.7940e-01, lr = 3.9063e-05
Epoch 500: Train Loss = 2.2208e-06, Val Loss = 2.4417e-06, Time = 5.1332

...

Epoch 0: Train Loss = 1.6937e-02, Val Loss = 1.4603e-03, Time = 4.9519e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 7.4143e-04, Val Loss = 3.2913e-04, Time = 5.4437e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.9058e-04, Val Loss = 1.0592e-04, Time = 5.7355e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.7895e-05, Val Loss = 5.7363e-05, Time = 5.1326e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 4.2875e-05, Val Loss = 3.8228e-05, Time = 4.8833e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.0216e-05, Val Loss = 2.7396e-05, Time = 7.4894e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.3790e-06, Val Loss = 3.2847e-06, Time = 4.3461e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.6235e-06, Val Loss = 2.6819e-06, Time = 5.0405e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.1906e-06, Val Loss = 2.2425e-06, Time = 4.4115e-01, lr = 5.0000e-03
Epoch 400: Train Loss = 1.9872e-06, Val Loss = 1.9949e-06, Time = 4.4202e-01, lr = 3.1250e-04
Epoch 500: Train Loss = 2.0357e-06, Val Loss = 1.9582e-06, Time = 4.6507

Fold 4:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.4567e-02, Val Loss = 1.9715e-03, Time = 5.3402e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 4.5445e-04, Val Loss = 2.0157e-04, Time = 5.7342e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.2479e-04, Val Loss = 7.7004e-05, Time = 4.6643e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.7442e-05, Val Loss = 5.5902e-05, Time = 4.6011e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 4.9894e-05, Val Loss = 4.2496e-05, Time = 4.7077e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.8309e-05, Val Loss = 3.2036e-05, Time = 4.5666e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.1616e-06, Val Loss = 3.1381e-06, Time = 5.1579e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.5493e-06, Val Loss = 2.5638e-06, Time = 4.4369e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 2.3287e-06, Val Loss = 2.2736e-06, Time = 4.6693e-01, lr = 2.5000e-03
Epoch 400: Train Loss = 2.1953e-06, Val Loss = 2.0411e-06, Time = 4.4330e-01, lr = 6.2500e-04
Epoch 500: Train Loss = 2.1053e-06, Val Loss = 2.0403e-06, Time = 4.5387

Fold 5:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.7934e-02, Val Loss = 5.3502e-04, Time = 7.0259e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 2.2274e-04, Val Loss = 1.3427e-04, Time = 5.7133e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.0020e-04, Val Loss = 8.3448e-05, Time = 6.1200e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 7.1414e-05, Val Loss = 6.6513e-05, Time = 5.4490e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.7715e-05, Val Loss = 5.3410e-05, Time = 5.2636e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.5183e-05, Val Loss = 3.9310e-05, Time = 5.3229e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.4364e-06, Val Loss = 3.2928e-06, Time = 4.8976e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.8909e-06, Val Loss = 2.9078e-06, Time = 5.5517e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 2.8590e-06, Val Loss = 2.6861e-06, Time = 4.9965e-01, lr = 3.1250e-04
Epoch 400: Train Loss = 2.6986e-06, Val Loss = 2.5893e-06, Time = 4.7649e-01, lr = 1.5625e-04
Epoch 500: Train Loss = 2.6240e-06, Val Loss = 2.6292e-06, Time = 5.1868

In [17]:
avg_best_val_losses, np.argmin(avg_best_val_losses)

([2.149840457020348e-06,
  1.4529735145580551e-06,
  1.834684915788135e-06,
  2.1752475358230836e-06,
  1.982547925243645e-06,
  2.263210873414729e-06],
 1)

In [18]:
# # --- Configuration ---
# k_folds = 5
# n_epochs = 1000  
# batch_size = 50 
# learning_rate = 1e-2
# print_every = 100
# _precision = 4
# random_state = 1234


# kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


# val_results = []
# train_results = []

# def get_fresh_model_components():
#     f = FeluSigmoidMLP(dims=[2,10,10,10,10,2],lower_bound=-1, upper_bound=-0.1)
#     g = GeluSigmoidMLP(dims=[4,10,10,10,10,2],lower_bound=0, upper_bound=1)
#     model = FTNODE(f,g).to(device)
#     return f, g, model 

# # ==========================================
# # K-Fold Loop
# # ==========================================
# for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
#     print(f'\n--- FOLD {fold+1}/{k_folds} ---')

#     # 1. Re-initialize Model & Optimizer for this fold
#     f_fold, g_fold, model_fold = get_fresh_model_components()
#     model_fold.train() 
    
#     loss_criteria = nn.MSELoss()
    

#     opt = torch.optim.Adam(
#         list(f_fold.parameters()) + list(g_fold.parameters()), 
#         lr=learning_rate
#     )
    
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#         opt, mode="min", factor=0.5, patience=10
#     )

#     # 2. Create DataLoaders for this fold
#     train_subsampler = SubsetRandomSampler(train_ids)
#     val_subsampler = SubsetRandomSampler(val_ids)

#     trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
#     valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

#     # 3. Training & Validation Loop
#     best_val_loss = float('inf')
#     fold_losses = []

#     for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
#         t1 = time.time()
        
#         # --- TRAINING ---
#         model_fold.train()
#         train_loss = 0.0
        
#         for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
#             ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
#             u_func = lambda t: ui_expanded

#             opt.zero_grad()
            
#             dXi_pred = model_fold(ti, Xi, u_func)
#             loss = loss_criteria(dXi, dXi_pred)
            
#             loss.backward()
#             opt.step()
            
#             train_loss += loss.item()
        
#         train_loss /= len(trainloader)

#         # --- VALIDATION ---
#         model_fold.eval()
#         val_loss = 0.0
#         with torch.no_grad():
#             for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
#                 ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
#                 u_func = lambda t: ui_expanded

#                 dXi_pred = model_fold(ti, Xi, u_func)
#                 loss = loss_criteria(dXi, dXi_pred)
#                 val_loss += loss.item()
        
#         val_loss /= len(valloader)
        
#         # --- SCHEDULER & LOGGING ---
#         epoch_time = time.time() - t1
        
#         # Step scheduler based on VALIDATION loss (standard practice)
#         scheduler.step(val_loss) 
#         cur_lr = opt.param_groups[0]['lr']

#         if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
#             print(
#                 f"Epoch {epoch}: "
#                 f"Train Loss = {train_loss:.{_precision}e}, "
#                 f"Val Loss = {val_loss:.{_precision}e}, "
#                 f"Time = {epoch_time:.{_precision}e}, "
#                 f"lr = {cur_lr:.{_precision}e}"
#             )

#         if val_loss < best_val_loss:
#             best_val_loss = val_loss
#             best_val_train_loss = train_loss
#             best_fold = fold
            
#     print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
#     val_results.append(best_val_loss)
#     train_results.append(best_val_train_loss)

# # --- SUMMARY ---
# print("\nK-Fold Cross Validation Results:")
# avg_loss = np.mean(val_results)
# avg_train_loss = np.mean(train_results)
# print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
# avg_best_val_losses.append(avg_loss)
# avg_best_train_losses.append(avg_train_loss)